In [ ]:
# CleanData.ipynb

import os
import re
import glob
from itertools import combinations
from collections import defaultdict
import pandas as pd

In [ ]:
from google.colab import files
uploaded = files.upload()  # chọn file zip metadata, ví dụ cit-HepTh-abstracts.zip

Saving cit-HepTh-abstracts.zip to cit-HepTh-abstracts.zip


In [ ]:
!unzip -q "cit-HepTh-abstracts.zip" -d "cit-HepTh-abstracts"

In [ ]:
root_dir = "cit-HepTh-abstracts"  # sau khi unzip, cd vào thư mục này
output_dir = "output_csv"

In [ ]:
# --- 1. HÀM XỬ LÝ NAME & AFFILIATION ---
def normalize_author_token(t):
    """
    Tách tên và affiliation.
    Input: "Paul J. Steinhardt (Princeton)"
    Output: "Paul J. Steinhardt", "Princeton"
    """
    t = t.strip()
    affiliation = None

    # Tìm nội dung trong ngoặc đơn cuối cùng
    match = re.search(r'\((.*?)\)', t)
    if match:
        affiliation = match.group(1).strip() # Lấy nội dung trong ngoặc
        t = re.sub(r'\(.*?\)', '', t).strip() # Xóa ngoặc đi

    # 1. Latex accents cleanup
    # Convert \"u, "u -> ue (German convention) to unify versions

    # Clean basic chars
    t = t.replace('\\', '').replace('"', '').replace('{', '').replace('}', '').replace("'", "")

    # Lowercase
    t = t.replace(r'\"a', 'ae').replace(r'"a', 'ae')
    t = t.replace(r'\"o', 'oe').replace(r'"o', 'oe')
    t = t.replace(r'\"u', 'ue').replace(r'"u', 'ue')

    # Uppercase
    t = t.replace(r'\"A', 'Ae').replace(r'"A', 'Ae')
    t = t.replace(r'\"O', 'Oe').replace(r'"O', 'Oe')
    t = t.replace(r'\"U', 'Ue').replace(r'"U', 'Ue')

    # Eszett
    t = t.replace(r'\ss', 'ss')

    # 2. Xóa ký tự lạ (backslash còn sót lại)
    t = t.replace('\\', '')

    # 3. Xóa dấu ngoặc kép còn sót (nếu có)
    t = t.replace('"', '')

    # 4. Chuẩn hóa dấu chấm: "A.B.Name" -> "A. B. Name"
    t = re.sub(r'\.([A-Za-z])', r'. \1', t)

    # 5. Xóa khoảng trắng thừa
    t = re.sub(r'\s+', ' ', t)

    # 6. Viết hoa chữ cái đầu (Title Case)
    t = t.title()

    # 7. Xóa phần đuôi rác (số trang dính ở cuối tên)
    # Ví dụ: "A.V. Galajinsky. 12 P." -> "A.V. Galajinsky"
    t = re.sub(r'[\.\s]+\d+\s*(pages?|p\.?)\s*$', '', t, flags=re.IGNORECASE)

    return t, affiliation

def split_authors_respecting_parens(raw_authors):
    """Tách tác giả theo dấu phẩy, nhưng BỎ QUA dấu phẩy nằm trong ngoặc"""
    s = raw_authors.replace(" and ", ",").replace("&", ",")
    tokens = []
    current_token = []
    paren_depth = 0
    for char in s:
        if char == '(': paren_depth += 1
        elif char == ')': paren_depth = max(0, paren_depth - 1)
        elif char == ',' and paren_depth == 0:
            tokens.append("".join(current_token))
            current_token = []
            continue
        current_token.append(char)
    if current_token: tokens.append("".join(current_token))
    return [t.strip() for t in tokens if t.strip()]

# --- 2. HÀM PARSE FILE .ABS ---
def parse_abs_text(text: str):
    lines = text.splitlines()
    metadata = {
        'paper_id': None, 'title': None, 'year': None,
        'raw_authors': None, 'authors_clean': [], 'affiliations_list': []
    }

    for i, line in enumerate(lines):
        line_clean = line.strip()

        # Parse Paper ID
        if line_clean.startswith("Paper:") or line_clean.startswith("arXiv:"):
            parts = line_clean.split(":", 1)
            if len(parts) > 1:
                metadata['paper_id'] = parts[1].strip()

        # Parse Year from Date
        if line_clean.startswith("Date:"):
            date_str = line_clean.split(":", 1)[1]
            year_match = re.search(r'\b(19|20)\d{2}\b', date_str)
            if year_match:
                metadata['year'] = int(year_match.group(0))

        # Parse Title (Multi-line)
        if line_clean.startswith("Title:"):
            title_str = line_clean.split(":", 1)[1].strip()
            j = i + 1
            while j < len(lines):
                next_line = lines[j]
                # Check if this line looks like a start of another field, even if indented (rare but possible)
                next_line_clean = next_line.strip()
                if any(next_line_clean.startswith(k + ":") for k in ["Comments", "Journal-ref", "Report-no", "Date", "Title", "Paper", "From"]):
                    break
                if re.match(r'^\d+\s*(pages?|p\b)', next_line_clean, re.IGNORECASE):
                    break
                if next_line.startswith(' ') or next_line.startswith('\t'):
                    title_str += " " + next_line.strip()
                    j += 1
                else:
                    break
            metadata['title'] = re.sub(r'\s+', ' ', title_str)

        # Parse Authors (Multi-line)
        if line_clean.startswith("Authors:") or line_clean.startswith("Author:"):
            author_str = line_clean.split(":", 1)[1].strip()
            j = i + 1
            while j < len(lines):
                next_line = lines[j]
                if any(next_line.strip().startswith(k + ":") for k in ["Comments", "Journal-ref", "Report-no", "Date", "Title"]):
                    break
                if next_line.startswith(' ') or next_line.startswith('\t'):
                    author_str += " " + next_line.strip()
                    j += 1
                else:
                    break
            metadata['raw_authors'] = author_str

    return metadata

# --- 3. HÀM CHÍNH TẠO DATAFRAME ---
def generate_csvs(root_dir, output_dir):
    files = glob.glob(os.path.join(root_dir, "**", "*.abs"), recursive=True)
    print(f"Processing {len(files)} files...")

    papers_rows = []
    paper_authors_rows = []

    for filepath in files:
        filename = os.path.basename(filepath)
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
        except:
            continue

        parsed = parse_abs_text(content)

        # Fallback ID & Year
        p_id = parsed['paper_id'] if parsed['paper_id'] else os.path.splitext(filename)[0]
        year = parsed['year']
        if not year:
             path_parts = filepath.split(os.sep)
             for part in path_parts:
                 if part.isdigit() and len(part) == 4:
                     year = int(part)
                     break

        # 1. PAPERS TABLE
        papers_rows.append({
            'paper_id': p_id,
            'year': year,
            'title': parsed['title'],
            'raw_authors': parsed['raw_authors']
        })

        # 2. PAPER_AUTHORS TABLE
        if parsed['raw_authors']:
            # FIX: Loại bỏ block định nghĩa affiliation ((1)..., (2)...) ở cuối
            raw_nodef = parsed['raw_authors'].split('((')[0]
            tokens = split_authors_respecting_parens(raw_nodef)
            for token in tokens:
                name, aff = normalize_author_token(token)
                # Safety filter: ignore if name looks like page count, Latex, or Affiliation leakage
                if re.match(r'^\d+(\s*([a-z]+|[.,]))*$', name, re.IGNORECASE):
                     continue
                if re.search(r'\b(latex|tex)\b', name, re.IGNORECASE):
                  continue
                if re.search(r'\b(University|Department|Institute|Physics|Laboratory)\b', name, re.IGNORECASE):
                  continue
                if len(name) > 1: # Filter rác
                    paper_authors_rows.append({
                        'paper_id': p_id,
                        'author': name,
                        'year': year,
                        'affiliation': aff
                    })

    # Tạo DataFrame
    df_papers = pd.DataFrame(papers_rows)[['paper_id', 'year', 'title', 'raw_authors']]
    df_paper_authors = pd.DataFrame(paper_authors_rows)

    # 3. AUTHORS TABLE (Derived)
    if not df_paper_authors.empty:
        authors_grouped = df_paper_authors.groupby('author').agg(
            first_year=('year', 'min'),
            n_papers=('paper_id', 'count'),
            affiliations_seen=('affiliation', lambda x: list(set(i for i in x if i)))
        ).reset_index()
    else:
        authors_grouped = pd.DataFrame()

    # 4. COAUTHOR_EDGES TABLE (Derived)
    edges = []
    if not df_paper_authors.empty:
        for pid, group in df_paper_authors.groupby('paper_id'):
            auths = sorted(group['author'].unique())
            if len(auths) > 1:
                year = group['year'].iloc[0]
                for a1, a2 in combinations(auths, 2):
                    edges.append({'author1': a1, 'author2': a2, 'year': year})

    if edges:
        df_edges = pd.DataFrame(edges).groupby(['author1', 'author2']).agg(
            first_year=('year', 'min'),
            weight=('year', 'count')
        ).reset_index()
    else:
        df_edges = pd.DataFrame()

    # Export CSV
    os.makedirs(output_dir, exist_ok=True)
    df_papers.to_csv(os.path.join(output_dir, 'papers.csv'), index=False)
    df_paper_authors.to_csv(os.path.join(output_dir, 'paper_authors.csv'), index=False)
    authors_grouped.to_csv(os.path.join(output_dir, 'authors.csv'), index=False)
    df_edges.to_csv(os.path.join(output_dir, 'coauthor_edges.csv'), index=False)

    print(f"Done! Check {output_dir}")

In [ ]:
generate_csvs(root_dir, output_dir)

Processing 29555 files...
Done! Check output_csv
